# Multilingual Speech Recognition Evaluation for Airport 

This notebook evaluates an automatic speech recognition pipeline for short airport-related phrases in English, Spanish, and Italian. It downloads the required Sherpa-ONNX models, prepares the reference transcripts, compares raw and preprocessed audio transcriptions, and reports word error rate (WER) by file and by language.

## Dependency Setup

The first code cell installs the pinned speech-recognition dependencies used throughout the notebook. Because many of the traditional models covered in the module (such as DeepSpeech) are no longer actively supported, I opted for sherpa-onnx as a modern, robust alternative. It provides the recognizer API, while onnxruntime handles the execution of the exported neural-network model files.

In [1]:
!python -m pip install sherpa-onnx==1.9.30 onnxruntime==1.14.1

You should consider upgrading via the '/opt/conda/bin/python -m pip install --upgrade pip' command.


In [2]:
import tarfile
import unicodedata
import urllib.error
import urllib.request
import wave
from pathlib import Path

import numpy as np
import sherpa_onnx
from scipy.signal import butter, sosfiltfilt

## Model Download Helpers

The recognizers use one English Zipformer model and two Kroko models for Spanish and Italian. The helper functions download missing files, keep partially downloaded files from being reused after an error, extract the English archive, and keep the smaller int8 ONNX weights.

In [3]:
MODELS_DIR = Path("models")

# English: a tar.bz2 from the sherpa-onnx GitHub releases.
EN_URL = (
    "https://github.com/k2-fsa/sherpa-onnx/releases/download/"
    "asr-models/sherpa-onnx-streaming-zipformer-en-2023-06-26.tar.bz2"
)
EN_DIR = MODELS_DIR / "sherpa-onnx-streaming-zipformer-en-2023-06-26"

# Spanish + Italian: individual Kroko files from Hugging Face.
KROKO_BASE = "https://huggingface.co/hudaiapa88/sherpa-stt-onnx/resolve/main"
KROKO_FILES = (
    "encoder.int8.onnx",
    "decoder.int8.onnx",
    "joiner.int8.onnx",
    "tokens.txt",
)
KROKO_LANGS = {"es": "kroko-es", "it": "kroko-it"}
KROKO_VARIANT = "kroko_64l"


def download(url, dest):
    """Stream a URL to dest with a progress line. Fails loudly on HTTP errors
    and cleans up the partial file, so a 404 never leaves a stub behind."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print("  skip (exists): {}".format(dest))
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(dest) + ".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "model-downloader"})
        with urllib.request.urlopen(req) as resp, open(tmp, "wb") as f:
            total = int(resp.headers.get("Content-Length", 0))
            read = 0
            while True:
                chunk = resp.read(1 << 20)  # 1 MB
                if not chunk:
                    break
                f.write(chunk)
                read += len(chunk)
                if total:
                    pct = 100.0 * read / total
                    print("\r  {}: {} / {} MB ({:4.1f}%)".format(
                        dest.name, read >> 20, total >> 20, pct), end="", flush=True)
        if total:
            print()
        tmp.rename(dest)
    except urllib.error.HTTPError as e:
        if tmp.exists():
            tmp.unlink()
        raise SystemExit("HTTP {} for {}".format(e.code, url))
    except BaseException:
        if tmp.exists():
            tmp.unlink()
        raise


def fetch_english():
    if EN_DIR.is_dir():
        print("English: already extracted at {}".format(EN_DIR))
        return
    archive = MODELS_DIR / "en.tar.bz2"
    print("English: downloading...")
    download(EN_URL, archive)
    print("English: extracting...")
    with tarfile.open(archive, "r:bz2") as tar:
        tar.extractall(MODELS_DIR)
    archive.unlink()
    # This release ships both fp32 and int8 weights. I will use the int8 models.
    for f in EN_DIR.glob("*.onnx"):
        if ".int8." in f.name:
            continue
        int8_twin = f.with_name(f.name.replace(".onnx", ".int8.onnx"))
        if int8_twin.exists():
            print("  removing fp32 (int8 kept): {}".format(f.name))
            f.unlink()
    print("English: ready at {}".format(EN_DIR))


def fetch_kroko():
    for lang, folder in KROKO_LANGS.items():
        print("{} (Kroko {}): downloading...".format(lang.upper(), KROKO_VARIANT))
        for name in KROKO_FILES:
            url = "{}/{}/{}/{}".format(KROKO_BASE, lang, KROKO_VARIANT, name)
            download(url, MODELS_DIR / folder / name)
        print("{}: ready at {}".format(lang.upper(), MODELS_DIR / folder))


In [4]:
MODELS_DIR.mkdir(exist_ok=True)
fetch_english()
fetch_kroko()

English: already extracted at models/sherpa-onnx-streaming-zipformer-en-2023-06-26
ES (Kroko kroko_64l): downloading...
  skip (exists): models/kroko-es/encoder.int8.onnx
  skip (exists): models/kroko-es/decoder.int8.onnx
  skip (exists): models/kroko-es/joiner.int8.onnx
  skip (exists): models/kroko-es/tokens.txt
ES: ready at models/kroko-es
IT (Kroko kroko_64l): downloading...
  skip (exists): models/kroko-it/encoder.int8.onnx
  skip (exists): models/kroko-it/decoder.int8.onnx
  skip (exists): models/kroko-it/joiner.int8.onnx
  skip (exists): models/kroko-it/tokens.txt
IT: ready at models/kroko-it


## Model Paths

The `MODELS` dictionary maps each language code used later in the notebook to the directory containing that language's ONNX encoder, decoder, joiner, and token files.

In [5]:
MODELS = {
    "EN": "./models/sherpa-onnx-streaming-zipformer-en-2023-06-26",
    "ES": "./models/kroko-es",
    "IT": "./models/kroko-it",
}

## Reference Transcripts

The reference dictionary contains the expected text for each provided audio file. These references are used later to measure recognition accuracy. `OWN_SENTENCES` adds two extra recordings that I did on my laptop

In [6]:
REFERENCES = {
    "EN/checkin.wav":        "Where is the check-in desk?",
    "EN/parents.wav":        "I have lost my parents.",
    "EN/suitcase.wav":       "Please, I have lost my suitcase.",
    "EN/what_time.wav":      "What time is my plane?",
    "EN/where.wav":          "Where are the restaurants and shops?",
    "EN/checkin_child.wav":  "Where is the check-in desk?",
    "EN/parents_child.wav":  "I have lost my parents.",
    "EN/suitcase_child.wav": "Please, I have lost my suitcase.",
    "EN/what_time_child.wav": "What time is my plane?",
    "EN/where_child.wav":    "Where are the restaurants and shops?",
 
    "IT/checkin_it.wav":     "Dove e' il bancone?",
    "IT/parents_it.wav":     "Ho perso i miei genitori.",
    "IT/suitcase_it.wav":    "Per favore, ho perso la mia valigia.",
    "IT/what_time_it.wav":   "A che ora e' il mio aereo?",
    "IT/where_it.wav":       "Dove sono i ristoranti e i negozi?",
 
    "ES/checkin_es.wav":     "¿Dónde están los mostradores?",
    "ES/parents_es.wav":     "He perdido a mis padres.",
    "ES/suitcase_es.wav":    "Por favor, he perdido mi maleta.",
    "ES/what_time_es.wav":   "¿A qué hora es mi avión?",
    "ES/where_es.wav":       "¿Dónde están los restaurantes y las tiendas?",
}

OWN_SENTENCES = [
    ("OWN/own_sentence1.wav", "EN", "Where I can find a taxi?"),
    ("OWN/own_sentence2.wav", "ES", "Necesito cambiar mi asiento."),
]

## Audio Preprocessing

The preprocessing step removes low-frequency noise with a high-pass filter, normalizes loudness using RMS energy, and adds a short deterministic low-amplitude lead-in. We later compares transcription with and without this cleaning step to show that preprocessing improves recognition.

In [7]:
def highpass(samples, sample_rate, cutoff_hz=80.0, order=4):
    sos = butter(order, cutoff_hz, btype="highpass", fs=sample_rate, output="sos")
    padlen = min(3 * (sos.shape[0] * 2), len(samples) - 1)
    return sosfiltfilt(sos, samples, padlen=max(padlen, 0)).astype(np.float32)

def rms_normalize(samples, target_dbfs=-20.0, peak_ceiling=0.97):
    rms = np.sqrt(np.mean(samples**2)) + 1e-9
    out = samples * (10.0 ** (target_dbfs / 20.0) / rms)
    peak = np.max(np.abs(out)) + 1e-9
    if peak > peak_ceiling:
        out = out * (peak_ceiling / peak)
    return out.astype(np.float32) 

def preprocess(samples, sample_rate, cutoff_hz=80.0, target_dbfs=-20.0, lead_sec=0.8):
    x = highpass(samples, sample_rate, cutoff_hz)
    x = rms_normalize(x, target_dbfs)
    pad = (np.random.RandomState(0).randn(int(lead_sec * sample_rate)) * 1e-4).astype(
        np.float32
    )
    return np.concatenate([pad, x])

## Recognition Utilities

`build_recognizer` creates a CPU-based Sherpa-ONNX streaming recognizer from a model directory. `read_wave` loads mono 16-bit PCM audio into floating-point samples. `transcribe` optionally preprocesses the samples, feeds them to the recognizer, appends trailing silence so the stream can finish, and returns the decoded text.

In [8]:
def build_recognizer(model_dir):
    d = Path(model_dir)
    return sherpa_onnx.OnlineRecognizer.from_transducer(
        tokens=str(d / "tokens.txt"),
        encoder=str(next(d.glob("encoder*.onnx"))),
        decoder=str(next(d.glob("decoder*.onnx"))),
        joiner=str(next(d.glob("joiner*.onnx"))),
        num_threads=2,
        provider="cpu",
        decoding_method="greedy_search",
    )


def read_wave(path):
    with wave.open(str(path), "rb") as wf:
        assert wf.getnchannels() == 1, "%s: must be mono" % path
        assert wf.getsampwidth() == 2, "%s: must be 16-bit PCM" % path
        rate = wf.getframerate()
        frames = wf.readframes(wf.getnframes())
    samples = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
    return samples, rate


def transcribe(recognizer, samples, rate, clean, tail_sec=2.0):
    if clean:
        samples = preprocess(samples, rate)
    s = recognizer.create_stream()
    s.accept_waveform(rate, samples)
    s.accept_waveform(rate, np.zeros(int(tail_sec * rate), np.float32))  # was 0.66
    s.input_finished()
    while recognizer.is_ready(s):
        recognizer.decode_stream(s)
    return recognizer.get_result(s)

## Audio Assets and Recognizers

The audio files are expected under `Ex4_files/Ex4_audio_files`, grouped by language, and we do online inference

In [ ]:
ASSETS = Path("assets")

In [10]:
recognizers = {lang: build_recognizer(path) for lang, path in MODELS.items()}

In [11]:
def collect_items():
    for lang in ("EN", "ES", "IT"):
        for wav in sorted((ASSETS / lang).glob("*.wav")):
            key = "%s/%s" % (lang, wav.name)
            if key in REFERENCES:
                yield lang, wav, REFERENCES[key]
    for rel, lang, ref in OWN_SENTENCES:
        p = ASSETS / rel
        if p.exists():
            yield lang, p, ref

## Raw vs. Clean Transcription

This loop transcribes every evaluation item twice: once using the original waveform and once using the preprocessing pipeline. Printing both outputs makes it easy to inspect where filtering and normalization help, such as recovering missing words or correcting a phrase.

In [12]:
for lang, path, ref in collect_items():
    raw = transcribe(recognizers[lang], *read_wave(path), clean=False)
    clean = transcribe(recognizers[lang], *read_wave(path), clean=True)

    print(f"\n╭─ {lang} · {path}")
    print(f"│ raw   {raw}")
    print(f"│ clean {clean}")
    print("╰" + "─" * 60)


╭─ EN · Ex4_files/Ex4_audio_files/EN/checkin.wav
│ raw   WHERE IS THE CHICKEN DESK
│ clean WHERE IS THE CHICKEN DESK
╰────────────────────────────────────────────────────────────

╭─ EN · Ex4_files/Ex4_audio_files/EN/parents.wav
│ raw   I HAVE LOST MY PARENTS
│ clean I HAVE LOST MY PARENTS
╰────────────────────────────────────────────────────────────

╭─ EN · Ex4_files/Ex4_audio_files/EN/suitcase.wav
│ raw   PLEASE I HAVE LOST MY SUIT CASE
│ clean PLEASE I HAVE LOST MY SUIT CASE
╰────────────────────────────────────────────────────────────

╭─ EN · Ex4_files/Ex4_audio_files/EN/what_time.wav
│ raw   WHAT TIME IS MY PLAYING
│ clean AT WHAT TIME IS MY PLANE
╰────────────────────────────────────────────────────────────

╭─ EN · Ex4_files/Ex4_audio_files/EN/where.wav
│ raw   THE RESTAURANTS AND SHOPS
│ clean WHERE ARE THE RESTAURANTS AND SHOPS
╰────────────────────────────────────────────────────────────

╭─ ES · Ex4_files/Ex4_audio_files/ES/checkin_es.wav
│ raw   ¿Dónde están los mostrado

## Word Error Rate

Before scoring, `normalize` lowercases text, removes accents, drops punctuation, and collapses whitespace so small formatting differences do not count as recognition errors. `wer` then computes word-level substitutions, deletions, and insertions with dynamic programming and returns the WER percentage.

In [13]:
def normalize(text):
    text = text.lower().replace("-", " ").replace("/", " ")
    text = "".join(
        c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn"
    )  # drop accents
    text = "".join(ch for ch in text if ch.isalnum() or ch.isspace())
    return " ".join(text.split())


def wer(reference, hypothesis):
    """WER = (S + D + I) / N * 100, word-level Levenshtein."""
    ref, hyp = normalize(reference).split(), normalize(hypothesis).split()
    # d[i][j] = (S, D, I) of the best alignment of ref[:i] vs hyp[:j]
    d = [[(0, 0, 0)] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(1, len(ref) + 1):
        d[i][0] = (0, i, 0)  # leftover ref words = deletions
    for j in range(1, len(hyp) + 1):
        d[0][j] = (0, 0, j)  # leftover hyp words = insertions
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            if ref[i - 1] == hyp[j - 1]:
                d[i][j] = d[i - 1][j - 1]  # match: carry counts forward
            else:
                s, dl, ins = d[i - 1][j - 1], d[i - 1][j], d[i][j - 1]
                d[i][j] = min(
                    (s[0] + 1, s[1], s[2]),  # substitution
                    (dl[0], dl[1] + 1, dl[2]),  # deletion
                    (ins[0], ins[1], ins[2] + 1),  # insertion
                    key=sum,
                )
    substitutions, deletions, insertions = d[len(ref)][len(hyp)]
    reference_word_count = len(ref)
    score = (
        100.0 * (substitutions + deletions + insertions) / reference_word_count
        if reference_word_count
        else (100.0 if hyp else 0.0)
    )
    return {
        "S": substitutions,
        "D": deletions,
        "I": insertions,
        "N": reference_word_count,
        "wer": score,
    }


## Final Accuracy Summary


In [14]:
rows, agg = [], {}
for lang, path, ref in collect_items():
    hyp = transcribe(recognizers[lang], *read_wave(path), clean=True)
    m = wer(ref, hyp)
    rows.append((lang, path.name, m))
    a = agg.setdefault(lang, dict(S=0, D=0, I=0, N=0))
    for k in a:
        a[k] += m[k]
    print(
        "%-4s %-22s S=%d D=%d I=%d N=%-2d WER=%5.1f%%"
        % (lang, path.name, m["S"], m["D"], m["I"], m["N"], m["wer"])
    )

print("\nPer-language corpus WER:")
for lang, a in agg.items():
    print("%-4s %5.1f%%" % (lang, 100.0 * (a["S"] + a["D"] + a["I"]) / a["N"]))


EN   checkin.wav            S=1 D=1 I=0 N=6  WER= 33.3%
EN   parents.wav            S=0 D=0 I=0 N=5  WER=  0.0%
EN   suitcase.wav           S=1 D=0 I=1 N=6  WER= 33.3%
EN   what_time.wav          S=0 D=0 I=1 N=5  WER= 20.0%
EN   where.wav              S=0 D=0 I=0 N=6  WER=  0.0%
ES   checkin_es.wav         S=0 D=0 I=0 N=4  WER=  0.0%
ES   parents_es.wav         S=0 D=0 I=0 N=5  WER=  0.0%
ES   suitcase_es.wav        S=0 D=0 I=0 N=6  WER=  0.0%
ES   what_time_es.wav       S=0 D=0 I=0 N=6  WER=  0.0%
ES   where_es.wav           S=0 D=0 I=0 N=7  WER=  0.0%
IT   checkin_it.wav         S=0 D=0 I=0 N=4  WER=  0.0%
IT   parents_it.wav         S=0 D=0 I=0 N=5  WER=  0.0%
IT   suitcase_it.wav        S=0 D=0 I=0 N=7  WER=  0.0%
IT   what_time_it.wav       S=0 D=0 I=0 N=7  WER=  0.0%
IT   where_it.wav           S=0 D=0 I=0 N=7  WER=  0.0%
EN   own_sentence1.wav      S=0 D=0 I=0 N=6  WER=  0.0%
ES   own_sentence2.wav      S=0 D=0 I=0 N=4  WER=  0.0%

Per-language corpus WER:
EN    14.7%
ES     0.0